# C06: Proyecto Integrador - Gestor de Archivos y Datos

## Objetivos

Este es el **proyecto integrador del nivel básico**. Aquí pondrás en práctica **todo** lo aprendido en los módulos C01 a C05:

- **C01**: tipos de datos, f-strings, operadores, conversiones de tipo.
- **C02**: control de flujo, `if`/`elif`/`else`, ciclos `for` y `while`.
- **C03**: estructuras de datos: listas, tuplas, diccionarios, conjuntos, compresión.
- **C04**: funciones, parámetros, valores de retorno, type hints (si aplicas).
- **C05**: archivos y excepciones: `pathlib`, lectura/escritura, `try`/`except`.

Al finalizar este módulo serás capaz de:

1. **Construir un CLI (interfaz de línea de comandos)** funcional que recibe una carpeta.
2. **Inventariar** todos los archivos `.csv`, `.json` y `.txt` dentro de subcarpetas.
3. **Leer y procesar** archivos CSV: calcular promedio, totales y conteos por categoría.
4. **Combinar** múltiples archivos JSON en un único dataset.
5. **Generar un reporte de resumen** en texto plano.
6. **Manejar errores** con elegancia: archivos inexistentes, datos corruptos y división por cero.
7. **Escribir los resultados** en un archivo de salida.

## Analogía inicial: El gerente de almacén

Imagina que eres el gerente de un gran almacén que recibe cajas de datos todos los días. Cada departamento manda sus archivos: ventas en `CSV`, clientes en `JSON` y notas en `TXT`, guardados en distintas carpetas y subcarpetas.

Tu trabajo no es abrir cada archivo a mano (sería imposible con cientos de ellos). Tu trabajo es construir una **máquina automática** que:

1. **Recorre el almacén** (todas las carpetas) y encuentra qué hay.
2. **Lee cada caja** y extrae la información importante.
3. **Resume** todo en un único reporte para el jefe.
4. **Avisa** cuando algo está roto (un archivo corrupto, una caja vacía) sin dejar de trabajar.

Esa máquina es: **un programa (CLI) que recibe una carpeta y produce un reporte**. Eso es exactamente lo que construirás aquí.

## Requisitos del proyecto

Construirás un **script en línea de comandos** llamado `gestor_archivos`. Su flujo general es:

```
                                                  ┌───────────────────────┐
   python gestor_archivos.py <carpeta>             │  INVENTARIO (paso 2)  │
   ───────────────────────────────▶            ┌─▶ │  Recorre subcarpetas  │
                                              │  └───────────────────────┘
   ┌───────────────────┐                      │  ┌───────────────────────┐
   │ 1. VALIDAR PATH    │      fasta la ruta   ├─▶ │  3. LEER CSVs         │
   │  ruta existe?      │──────▶ que recorre el│  │  limpiar y tipificar   │
   └───────────────────┘                      │  └───────────────────────┘
                                              │  ┌───────────────────────┐
   ┌───────────────────┐                      ├─▶ │  4. ESTADÍSTICAS      │
   │  8. REPORTE FINAL  │◀─────── resultados   │  │  promedio / total     │
   │  escrito en TXT    │       combinados     │  └───────────────────────┘
   └───────────────────┘                      │  ┌───────────────────────┐
                                              ├─▶ │  5. COMBINAR JSONs    │
                                              │  └───────────────────────┘
                                              │  ┌───────────────────────┐
                                              └─▶ │  6. GENERAR REPORTE  │
                                                 └───────────────────────┘
```

### Qué debe hacer el programa

1. **Recibir una carpeta** desde la línea de comandos (`sys.argv`).
2. **Inventariar** todos los `.csv`, `.json` y `.txt` dentro de subcarpetas (recursivo).
3. **Leer y procesar** cada CSV: limpia valores vacíos, convierte a números.
4. **Calcular estadísticas** de los CSV: promedio, total, conteo por categoría.
5. **Combinar** los datos de todos los JSON en un único dataset.
6. **Generar un reporte de resumen** en texto, con los hallazgos de todo lo anterior.
7. **Manejar errores**: archivo inexistente, datos corruptos, división por cero.
8. **Escribir** el reporte en un archivo de salida.

## Preparación: Crear los datos de ejemplo

Antes de construir la "máquina", necesitamos **material de trabajo**: unos archivos de ejemplo. Los crearemos aquí mismo con Python, para que el proyecto sea 100% reproducible dentro del notebook.

Crearemos esta estructura dentro de tu carpeta temporal (usando `pathlib`):

```
carpeta_demo/
├── ventas/
│   ├── enero.csv
│   ├── febrero.csv
│   └── notas.txt
├── clientes/
│   ├── clientes.json
│   └── productos.json
└── corrupto.csv      ← archivo con datos malformados
```

In [ ]:
from pathlib import Path
import tempfile

# 1) Ruta base de la carpeta demo, dentro del directorio temporal del sistema
base = Path(tempfile.gettempdir()) / "carpeta_demo"
base.mkdir(parents=True, exist_ok=True)

# 2) Subcarpetas
(base / "ventas").mkdir(exist_ok=True)
(base / "clientes").mkdir(exist_ok=True)

# 3) CSV de ventas (enero)
(base / "ventas" / "enero.csv").write_text(
    "categoria,producto,precio,cantidad\n"
    "electronica,Teclado,25,2\n"
    "electronica,Mouse,15,5\n"
    "oficina,Resma papel,8,10\n"
    "oficina,Lapicero,2,50\n",
    encoding="utf-8",
)

# 4) CSV de ventas (febrero): incluye una fila vacía y una fila con valor vacío
(base / "ventas" / "febrero.csv").write_text(
    "categoria,producto,precio,cantidad\n"
    "electronica,Monitor,120,1\n"
    "oficina,Resma papel,8,12\n"
    "",
    encoding="utf-8",
)

# 5) Notas de texto
(base / "ventas" / "notas.txt").write_text(
    "Ventas estables a fin de mes.\nRevisar stock de monitores.\n",
    encoding="utf-8",
)

# 6) JSON de clientes y productos
(base / "clientes" / "clientes.json").write_text(
    '{"clientes": [{"id": 1, "nombre": "Ana", "ciudad": "Lima"},'
    '{"id": 2, "nombre": "Luis", "ciudad": "Bogota"}]}',
    encoding="utf-8",
)

(base / "clientes" / "productos.json").write_text(
    '{"productos": [{"id": 101, "nombre": "Teclado"},'
    '{"id": 102, "nombre": "Monitor"}]}',
    encoding="utf-8",
)

# 7) Archivo corrupto (JSON inválido)
(base / "clientes" / "corrupto.json").write_text(
    '{esto no es json valido',
    encoding="utf-8",
)

print("Carpeta de ejemplo creada en:", base)

## Diagrama del pipeline que vamos a construir

El proyecto completo se construye en **8 pasos**. Cada paso es una función pequeña y clara. El diagrama muestra cómo se conectan:

```
 pasos 1-2                pasos 3-5                pasos 6-7
 ┌──────────────┐    ┌────────────────────┐    ┌──────────────────┐
 │ validar_path │    │  leer_csv_limpio   │    │ generar_reporte  │
 │  inventariar │    │  calcular_stats    │    │      main()      │
 └──────────────┘    │  combinar_json     │    └──────────────────┘
        │            └────────────────────┘             │
        │  lista de        │   dataset                 │  texto
        ▼        archivos  ▼   combinado               ▼  reporte
   ┌──────────┐  ┌──────────────┐            ┌─────────────────────┐
   │ CARPETA  │─▶│  PROCESAR    │───────────▶│ reporte_<fecha>.txt │
   └──────────┘  └──────────────┘            └─────────────────────┘
```

## Paso 1: Configurar paths con `pathlib` y verificar existencia

La entrada de nuestro CLI será una **ruta de carpeta**. Lo primero es **validar** que exista y que sea un directorio. Esto evita errores misteriosos más adelante.

Aquí usamos:
- `Path()` de `pathlib` para representar rutas.
- `path.exists()` y `path.is_dir()` para verificar.
- `raise` para lanzar una excepción si la ruta no es válida.

In [ ]:
from pathlib import Path

def validar_ruta(ruta: str) -> Path:
    """Convierte una cadena a Path y verifica que exista y sea una carpeta.

    Argumentos:
        ruta (str): Ruta al directorio a procesar.

    Retorna:
        Path: Un objeto Path válido que apunta a un directorio existente.

    Lanza:
        FileNotFoundError: Si la ruta no existe.
        NotADirectoryError: Si existe pero no es una carpeta.
    """
    p = Path(ruta)
    if not p.exists():
        raise FileNotFoundError(f"La ruta no existe: {p}")
    if not p.is_dir():
        raise NotADirectoryError(f"La ruta no es una carpeta: {p}")
    return p

# Probamos con la carpeta de ejemplo
try:
    ruta_ok = validar_ruta(base)
    print("Ruta válida:", ruta_ok)
except (FileNotFoundError, NotADirectoryError) as e:
    print("Error capturado:", e)

## Paso 2: Función para inventariar archivos

Ahora recorremos **todas** las subcarpetas y agrupamos los archivos por extensión. Usamos `path.rglob()` de `pathlib`, que busca de forma **recursiva**.

Retornaremos un diccionario donde la clave es la extensión (sin punto) y el valor es la lista de `Path` de los archivos encontrados.

In [ ]:
EXTENSIONES = {"csv", "json", "txt"}

def inventariar(carpeta: Path) -> dict:
    """Recorre recursivamente una carpeta y agrupa archivos por extensión.

    Argumentos:
        carpeta (Path): Directorio raíz que se quiere inventariar.

    Retorna:
        dict: Mapeo {extensión: [Path, ...]} con los archivos encontrados.
    """
    resultado: dict = {ext: [] for ext in EXTENSIONES}
    for archivo in carpeta.rglob("*"):
        if archivo.is_file():
            ext = archivo.suffix.lstrip(".").lower()
            if ext in EXTENSIONES:
                resultado[ext].append(archivo)
    return resultado

inventario = inventariar(ruta_ok)
for ext, archivos in inventario.items():
    print(f".{ext}: {len(archivos)} archivo(s)")
    for a in archivos:
        print("   -", a.relative_to(ruta_ok))

## Paso 3: Función para leer y limpiar CSV

Los CSV suelen venir **sucios**: filas vacías, valores faltantes, números como texto. Aquí:

1. Leemos el archivo línea por línea.
2. Saltamos la cabecera (primera línea).
3. Filtramos líneas vacías.
4. Convierto `precio` y `cantidad` a `float`/`int` con `try/except`.

Retornamos una **lista de diccionarios** (cada fila es un dict), que es una estructura natural y fácil de procesar.

In [ ]:
def leer_csv_limpio(archivo: Path) -> list:
    """Lee un CSV y retorna una lista de diccionarios con datos limpios.

    Argumentos:
        archivo (Path): Ruta al archivo CSV.

    Retorna:
        list: Lista de filas (cada una es un dict con categoria, producto,
              precio y cantidad como números).
    """
    filas = []
    lineas = archivo.read_text(encoding="utf-8").strip().splitlines()

    if not lineas:
        return filas

    # La primera línea es la cabecera
    cabecera = lineas[0].split(",")

    for linea in lineas[1:]:
        if not linea.strip():
            continue  # saltar líneas vacías

        valores = linea.split(",")
        # Convertir a los tipos correspondientes con manejo de errores
        try:
            precio = float(valores[2].strip())
            cantidad = int(valores[3].strip())
        except (IndexError, ValueError):
            continue  # fila corrupta: la omitimos con seguridad

        filas.append(
            {
                cabecera[0].strip(): valores[0].strip(),
                cabecera[1].strip(): valores[1].strip(),
                "precio": precio,
                "cantidad": cantidad,
            }
        )

    return filas

# Probamos con los dos CSV de ventas
for csv in inventario["csv"]:
    filas = leer_csv_limpio(csv)
    print(f"{csv.name}: {len(filas)} fila(s)")
    for f in filas:
        print("   ", f)

## Paso 4: Función para computar estadísticas

Ahora que tenemos las filas limpias, calculamos métricas útiles por archivo:

- **Total de ventas** = suma de `precio * cantidad`.
- **Promedio** de precio = total / cantidad de productos (con `try/except` por división por cero).
- **Conteo por categoría** = cuántos productos hay por cada `categoria`.

Aquí es clave el manejo de la **división por cero**: si no hay productos, el promedio debe ser 0, no un error.

In [ ]:
def calcular_estadisticas(filas: list) -> dict:
    """Calcula total, promedio y conteo por categoría de una lista de filas.

    Argumentos:
        filas (list): Lista de dicts con 'categoria', 'precio' y 'cantidad'.

    Retorna:
        dict: Con las claves 'total', 'promedio', 'n_productos' y
              'por_categoria'.
    """
    total = sum(f["precio"] * f["cantidad"] for f in filas)
    n = len(filas)

    # División por cero manejada con try/except
    try:
        promedio = total / n
    except ZeroDivisionError:
        promedio = 0.0

    # Conteo por categoría con un diccionario acumulador
    por_categoria: dict = {}
    for f in filas:
        cat = f["categoria"]
        por_categoria[cat] = por_categoria.get(cat, 0) + 1

    return {
        "total": total,
        "promedio": promedio,
        "n_productos": n,
        "por_categoria": por_categoria,
    }

# Probamos con las filas de un CSV de ejemplo
filas_ejemplo = leer_csv_limpio(inventario["csv"][0])
stats = calcular_estadisticas(filas_ejemplo)
print("Estadísticas de", inventario["csv"][0].name)
print(f"  Total ventas: {stats['total']}")
print(f"  Promedio precio: {stats['promedio']:.2f}")
print(f"  Productos: {stats['n_productos']}")
print(f"  Por categoría: {stats['por_categoria']}")

## Paso 5: Función para combinar JSON

Los archivos JSON suelen tener una estructura de **clave → lista**. Queremos juntar todas esas listas en un único dataset, **aunque un archivo esté corrupto** (lo omitimos con un `try/except`).

La estrategia:

- Leer cada archivo con `json.load`.
- Si es un dict, **mezclar** sus listas de valores en un único dataset con la clave de origen.
- Si el archivo está corrupto, registrar el error y **continuar** con los demás.

In [ ]:
import json

def combinar_json(archivos: list) -> tuple:
    """Combina múltiples archivos JSON en un único dataset.

    Argumentos:
        archivos (list): Lista de objetos Path a archivos JSON.

    Retorna:
        tuple: (dataset combinado, lista de errores).
    """
    dataset: dict = {}
    errores = []

    for archivo in archivos:
        try:
            with archivo.open(encoding="utf-8") as f:
                datos = json.load(f)
        except (json.JSONDecodeError, OSError) as e:
            errores.append(f"{archivo.name}: {e}")
            continue

        if isinstance(datos, dict):
            for clave, valor in datos.items():
                # Acumulamos listas bajo la misma clave
                if isinstance(valor, list):
                    dataset.setdefault(clave, []).extend(valor)
                else:
                    dataset.setdefault(clave, []).append(valor)

    return dataset, errores

dataset, errores = combinar_json(inventario["json"])
print("Dataset combinado:")
for clave, valores in dataset.items():
    print(f"  {clave}: {len(valores)} registro(s)")
for e in errores:
    print("  [ERROR]", e)

## Paso 6: Función para generar el reporte

Ahora reunimos **todo** en un texto legible que será nuestro reporte. Construimos el texto con f-strings y luego lo **escribimos** en un archivo `.txt` de salida con `pathlib`.

El reporte incluirá:
- Los totales y promedios por CSV.
- El conteo por categoría.
- El resumen del dataset JSON combinado.
- Los errores encontrados (para que el usuario se entere de qué se omitió).

In [ ]:
def generar_reporte(
    carpeta: Path,
    inventario: dict,
    stats_csv: dict,
    dataset_json: dict,
    errores: list,
) -> str:
    """Genera un reporte de resumen como texto plano.

    Argumentos:
        carpeta: Directorio procesado.
        inventario: Resultado de la función `inventariar`.
        stats_csv: Mapeo {archivo: estadísticas} de los CSV.
        dataset_json: Dataset JSON combinado.
        errores: Lista de errores encontrados durante el proceso.

    Retorna:
        str: Texto del reporte listo para escribir en un archivo.
    """
    lineas = []
    lineas.append("=" * 60)
    lineas.append("REPORTE DEL GESTOR DE ARCHIVOS")
    lineas.append("=" * 60)
    lineas.append(f"Carpeta analizada: {carpeta}")
    lineas.append("")

    total_archivos = sum(len(v) for v in inventario.values())
    lineas.append(f"Total de archivos encontrados: {total_archivos}")
    lineas.append(f"  CSV:  {len(inventario['csv'])}")
    lineas.append(f"  JSON: {len(inventario['json'])}")
    lineas.append(f"  TXT:  {len(inventario['txt'])}")
    lineas.append("")

    lineas.append("--- Estadísticas de ventas (CSV) ---")
    for archivo, s in stats_csv.items():
        lineas.append(f"  {archivo.name}:")
        lineas.append(f"    Total ventas: {s['total']}")
        lineas.append(f"    Promedio precio: {s['promedio']:.2f}")
        lineas.append(f"    Productos: {s['n_productos']}")
        lineas.append(f"    Por categoría: {s['por_categoria']}")

    lineas.append("")
    lineas.append("--- Datos combinados (JSON) ---")
    for clave, valores in dataset_json.items():
        lineas.append(f"  {clave}: {len(valores)} registro(s)")

    lineas.append("")
    lineas.append("--- Errores encontrados ---")
    if errores:
        for e in errores:
            lineas.append(f"  [ERROR] {e}")
    else:
        lineas.append("  Sin errores.")

    lineas.append("")
    lineas.append("=" * 60)
    lineas.append("Fin del reporte")
    return "\n".join(lineas)

# Construimos estadísticas por CSV para probar
stats_csv = {}
for csv in inventario["csv"]:
    stats_csv[csv] = calcular_estadisticas(leer_csv_limpio(csv))

reporte = generar_reporte(ruta_ok, inventario, stats_csv, dataset, errores)
print(reporte)

## Paso 7: Función `main()` y CLI (`sys.argv`)

Hasta ahora usamos las funciones por separado. Para que esto sea un **programa de línea de comandos**, necesitamos una función `main()` que:

1. Lea el argumento de la carpeta desde `sys.argv`.
2. Si no se pasa ninguna, use la carpeta de ejemplo por defecto.
3. **Orqueste** todos los pasos en el orden correcto.
4. Escriba el reporte en un archivo `reporte_<nombre>.txt`.
5. Capture los errores de alto nivel y muestre un mensaje amigable.

El flujo se ve así:

```
 sys.argv ──▶ main()
               │
               ├─▶ validar_ruta()
               ├─▶ inventariar()
               ├─▶ leer_csv_limpio() + calcular_estadisticas()
               ├─▶ combinar_json()
               ├─▶ generar_reporte()
               └─▶ Path.write_text()  →  reporte.txt
```

In [ ]:
import sys

def main(argv: list | None = None) -> None:
    """Punto de entrada principal del programa CLI.

    Uso desde terminal:
        python gestor_archivos.py [carpeta]

    Argumentos:
        argv (list | None): Argumentos de la línea de comandos. Si es None,
            usa sys.argv. El primer elemento útil (índice 1) es la carpeta.

    El reporte se escribe en 'reporte_<carpeta>.txt' dentro del directorio
    actual.
    """
    if argv is None:
        argv = sys.argv

    # Determinar la carpeta a procesar
    if len(argv) > 1:
        carpeta_input = argv[1]
    else:
        carpeta_input = str(base)  # por defecto, la carpeta de ejemplo

    try:
        carpeta = validar_ruta(carpeta_input)
    except FileNotFoundError as e:
        print(f"ERROR: {e}")
        return
    except NotADirectoryError as e:
        print(f"ERROR: {e}")
        return

    # 1) Inventariar
    inv = inventariar(carpeta)

    # 2) Procesar CSV → estadísticas
    stats_csv = {}
    for csv in inv["csv"]:
        stats_csv[csv] = calcular_estadisticas(leer_csv_limpio(csv))

    # 3) Combinar JSON (incluye manejo de archivos corruptos)
    dataset_json, errores = combinar_json(inv["json"])

    # 4) Generar y escribir el reporte
    reporte = generar_reporte(carpeta, inv, stats_csv, dataset_json, errores)
    salida = Path("reporte_gestor.txt")
    salida.write_text(reporte, encoding="utf-8")

    print("Proceso completado.")
    print("Reporte escrito en:", salida.resolve())


# Ejecutamos el flujo completo (sin argumentos reales de terminal)
if __name__ == "__main__":
    main([])

## Paso 8: Probar el flujo completo

Jupyter **no** pasa `sys.argv` automáticamente, así que probamos `main()` de dos formas:

1. Con la ruta de la carpeta de ejemplo pasada como argumento.
2. Simulando una llamada desde terminal con `sys.argv`.

Esto verifica que nuestro CLI funciona como un programa real.

In [ ]:
# Forma 1: pasar la carpeta como argumento a main()
main(["prog", str(base)])

# Mostramos el contenido del reporte generado
print()
print(Path("reporte_gestor.txt").read_text(encoding="utf-8"))

In [ ]:
# Forma 2: simular una llamada desde terminal con sys.argv
sys.argv = ["gestor_archivos.py", str(base)]
main()

# Y probamos un caso de error: carpeta inexistente
print()
sys.argv = ["gestor_archivos.py", str(base / "no_existe")]
main()

## Diagrama general del pipeline completo

Aquí tienes el mapa completo de lo que acabas de construir, con la entrada y salida de cada etapa:

```
┌──────────────┐
│  sys.argv    │  "carpeta_demo/"
│  (CLI)       │
└──────┬───────┘
       │
       ▼
┌──────────────┐   existe?  ──NO──▶  ❌ FileNotFoundError
│ validar_ruta │   es_dir?  ──NO──▶  ❌ NotADirectoryError
└──────┬───────┘
       ▼ Path válido
┌──────────────┐
│  inventariar │  rglob('*')
└──────┬───────┘
       │  { csv: [...], json: [...], txt: [...] }
       ▼
┌───────────────────────────────┐     ┌─────────────────────────────┐
│  leer_csv_limpio + calcular   │     │      combinar_json          │
│  promedio / total / categoría │     │  mezclar listas + omitir │
└──────────────┬────────────────┘     │  corruptos                 │
               │                       └──────────────┬──────────────┘
               ▼                       🟡 errores     ▼
   { archivo: {total, promedio, ...} }     (dataset, errores)
               \                           /
                ▼                         ▼
   ┌────────────────────────────────────────────┐
   │            generar_reporte()               │
   │   junta todo en un texto de resumen        │
   └───────────────────┬────────────────────────┘
                       ▼
   ┌────────────────────────────────────────────┐
   │        write_text -> reporte_gestor.txt    │
   └────────────────────────────────────────────┘
```

## Mejoras y extensiones

Tu gestor ya funciona, pero un proyecto real iría mucho más lejos. Algunas ideas para extenderlo:

| Mejora | Qué lograrías | Concepto asociado |
|--------|----------------|-------------------|
| Leer archivos con `csv.DictReader` | Robustez ante formatos más complejos | Módulo `csv` estándar |
| Generar el reporte en Markdown/HTML | Reportes más legibles y presentables | Formateo de texto |
| Ordenar archivos por fecha de modificación | Priorizar los más recientes | `path.stat().st_mtime` |
| Agregar opciones con `argparse` | CLI profesional con flags (`--salida`, `--formato`) | Módulo estándar `argparse` |
| Agrupar estadísticas en un **resumen global** | Vista consolidada de todos los CSV | Acumuladores transversales |
| Serializar el reporte a JSON | Interoperabilidad con otras herramientas | `json.dump` |
| Utilizar `functools.reduce` | Combinar datasets de forma funcional | Programación funcional |
| Registrar logs en vez de `print` | Trazabilidad de ejecuciones largas | Módulo `logging` |

**Sugerencia**: el orden natural de crecimiento es empezar por `argparse` y por el resumen global, porque dan más flexibilidad de uso.

### Resumen global (mini-muestra)

Mira cómo juntar **todos** los CSV en un solo promedio con un acumulador:

```python
total_global = sum(s["total"] for s in stats_csv.values())
print(f"Gran total de todas las ventas: {total_global}")
```

## Ejercicios extra: 2 retos

Pon a prueba lo aprendido. Intenta resolverlos **sin mirar** las respuestas.

### Reto 1: Resumen global por categoría

Extiende el pipeline para que, además de las estadísticas por archivo, calcule un **conteo global por categoría** cruzando todos los CSV. Pista: acumula el diccionario `por_categoria` de cada archivo.

```python
# Escribe tu solución aquí
def resumen_global(stats_csv: dict) -> dict:
    global_cat = {}
    for s in stats_csv.values():
        for cat, conteo in s["por_categoria"].items():
            global_cat[cat] = global_cat.get(cat, 0) + conteo
    return global_cat

print(resumen_global(stats_csv))
```

### Reto 2: Añadir promedio de cantidad

Los CSV guardan `precio` y `cantidad`. Construye una función que, dado un archivo CSV, retorne el **promedio de cantidades** vendidas por categoría (no de precios). Pista: agrupa por `categoria` y acumula `cantidad`.

```python
# Escribe tu solución aquí
def promedio_cantidad_por_categoria(archivo: Path) -> dict:
    acumulador = {}
    for f in leer_csv_limpio(archivo):
        cat = f["categoria"]
        datos = acumulador.setdefault(cat, {"suma": 0, "n": 0})
        datos["suma"] += f["cantidad"]
        datos["n"] += 1
    return {k: v["suma"] / v["n"] for k, v in acumulador.items()}

print(promedio_cantidad_por_categoria(inventario["csv"][0]))
```

## Resumen y conclusiones

¡Felicidades! Construiste un **proyecto integrador completo**. Repasemos qué habilidades pusiste en juego:

| Módulo | Habilidad aplicada en el proyecto |
|--------|-----------------------------------|
| C01 | Tipos, conversión a `float`/`int`, f-strings en el reporte |
| C02 | `if`/`else`, `while`/`for`, `continue` al saltar filas vacías |
| C03 | Listas de dicts, diccionarios acumuladores, `setdefault`, `get` |
| C04 | Funciones, parámetros, retorno, docstrings, type hints |
| C05 | `pathlib`, `read_text`/`write_text`, `json`, `try`/`except` |

### Conceptos clave aprendidos

1. **La arquitectura por funciones**: cada paso es una función pequeña y reutilizable (`inventariar`, `leer_csv_limpio`, `calcular_estadisticas`, `combinar_json`, `generar_reporte`, `main`).
2. **El manejo de errores defensivo**: no dejamos que un archivo corrupto derribe todo el programa; lo **omitimos** y lo reportamos.
3. **El flujo pipeline**: entrada (carpeta) → transformaciones → salida (reporte), con datos pasando de una etapa a la siguiente.
4. **Un CLI real**: `main()` + `sys.argv` convierten funciones sueltas en un programa usable desde la terminal.
5. **Escribir resultados**: `Path.write_text()` para persistir el reporte.

### Próximos pasos

Con este proyecto terminaste el **nivel básico de Python**. Ya tienes las bases sólidas para dar el salto a librerías de datos profesionales como **NumPy**, **Pandas** y **Matplotlib**, que automatizan mucho de lo que aquí hiciste a mano.